# Loading our Data

In [1]:
import pandas as pd 

df = pd.read_csv("IMDB Dataset.csv")

df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [2]:
df.shape

(50000, 2)

In [3]:
df.drop_duplicates(inplace = True)

df.shape

(49582, 2)

# Text Preprocessing

In [4]:
# !. Converting all data to lowercase.
# 2. removing URL, punctuations, HTML tags.
# 3. removes stopwords (a, an, is, the, etc.)  for this we have a pre built library has all stopwards for English.
# 4. Stemming (root form conversion).
# 5. Encoding target values 
# 6. Vectorization (TF-IDF)

In [5]:
# 1. Converting to lowercase.

df["review"] = df["review"].str.lower()

In [6]:
# 2(a). removing URL
# to do this we use re library 

import re 

# sample_text = "abc is the word, abc." # now replace abc => xyz
# new_text = re.sub("abc", "xyz", sample_text)
# new_text

def remove_urls(text):
    text = re.sub(r"http\S+" , "", text) # (pattern(to replace),replacement text, file in which to replace)
    # http\S+ means http + more text like https://google.com so S+ here is s://google.com

    # in re we have \S -> non white space character(only character no space) and \s -> white space character
    return text
 
df["review"] = df["review"].apply(remove_urls) # all the URLs removed

In [7]:
# 2(b). removing punctuations

def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # punctuations are the word that are not A-Z, a-z, 0-9, \s (spaces)
    # here we want to remove all the text other than A-Z, a-z, 0-9, \s.So, we put them in a sq bracket and ^ in start means we want to remove all text exluding these.

    return text

df["review"] = df["review"].apply(remove_punctuations)

In [8]:
# 2(c) Removing HTML tags
def remove_tags(text):
    text = re.sub(r"<.*?>" , "", text)  # . means any character b/w <> 
    # * means zero or more character
    # ? means remove until you got first > in the text
    return text

df["review"] = df["review"].apply(remove_tags)

In [9]:
# 3. Removing Stopwords

import nltk

# nltk -> natural language toolkit
# used to perform NLP operations and mainly used for Text processing

# before detecting stopwords we have to do tokenization.

nltk.download("punkt")     # punkt is the tokenizer in the nltk
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\SAMSUNG\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\SAMSUNG\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\SAMSUNG\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [11]:
# sample_text = "I like coding in python!"
# tokens = word_tokenize(sample_text)

# tokens  output -> ['I', 'like', 'coding', 'in', 'python', '!']

In [12]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text


df["review"] = df["review"].apply(remove_stopwords)

In [13]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


In [14]:
# 4. Stemming

# We will use PorterStemming for stemming

from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []  
    
    # Now for stemming we need individual words. So perform tokenization

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)

        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [15]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


In [16]:
# 5. Encoding 
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

y = df["sentiment"]
y  # 1 -> positive sentiment, 2 -> negative sentiment

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

In [17]:
# 6. Vectorization

from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features = 5000)

X = tf.fit_transform(df["review"])

# Converting Data into Tensors

In [18]:
from sklearn.model_selection import train_test_split 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

In [19]:
X_train.shape

(39665, 5000)

In [20]:
X_test.shape

(9917, 5000)

In [21]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [22]:
import torch 
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

X_train_tensor = torch.tensor(X_train, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype = torch.float32)

X_test_tensor = torch.tensor(X_test, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype = torch.float32)

In [23]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [24]:
trainloader = DataLoader(train_dataset, batch_size = 64, shuffle = True)
testloader = DataLoader(test_dataset, batch_size = 64, shuffle = True)

# Build RNN

In [33]:
import torch.optim as optim

# Many to One architecture

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size = 128, num_layers = 1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(
            input_size,
            hidden_size,
            num_layers,
            batch_first = True  # this reshape our input into a suitable format for RNN in one go.
        )

        # fully connected layer

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional step => (no. of layers, batch_size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out,_ = self.rnn(x, h0)  
        # 1st value = hidden state of all the timestamps => [batch, sequence_length(no. of timestamp, hidden_size]
        # 2nd value = final hidden state of last timestamp

        out = self.fc(out[:, -1, :]) # gives the last timestamp value 
        return out

In [34]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()

optimizer = optim.Adam(model.parameters())

# Training RNN


In [35]:
# unsqueeze -> add one dimension in data
# squeeze -> remove one dimension from data

epochs = 10

for epoch in range(epochs):
    model.train()

    for xb,yb in trainloader:
        optimizer.zero_grad()

        xb = xb.unsqueeze(1) 
        # we apply it bcz our tf-idf matrix is 2D and our model expects a 3D input.

        outputs = model(xb)  # output is (batch_size, 1) -> 2D2   [Forward propagation]

        outputs = torch.sigmoid(outputs.squeeze()) # sigmoid expects input to be in 1D so we have to squeeze one dimension.

        loss = criterion(outputs, yb)    # [Loss Function]

        loss.backward()                  # [Backward propagation]
        optimizer.step()                 # [Weight Updation]

    print(f"{epoch + 1} epoch loss is {loss.item()}")

1 epoch loss is 0.4128398895263672
2 epoch loss is 0.29371100664138794
3 epoch loss is 0.29678696393966675
4 epoch loss is 0.23021985590457916
5 epoch loss is 0.20072343945503235
6 epoch loss is 0.24706809222698212
7 epoch loss is 0.1548302173614502
8 epoch loss is 0.364324152469635
9 epoch loss is 0.24668240547180176
10 epoch loss is 0.18831412494182587


# Evaluation

In [36]:
model.eval()
with torch.no_grad():
    correct_vals = 0
    total_vals = 0
    for xb,yb in testloader:
        xb = xb.unsqueeze(1)

        outputs = model(xb)
        predicted_vals = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        correct_vals += (predicted_vals == yb).sum().item()
        total_vals += yb.size(0)

    print(f"{correct_vals} are correct out of {total_vals}. So, accuracy is {(correct_vals/total_vals)*100}%")

8482 are correct out of 9917. So, accuracy is 85.52989815468388%
